In [2]:
from pathlib import Path
import json
import os
import sys
import subprocess
import importlib

REQ_FILE = Path("requirements.txt") if Path("requirements.txt").exists() else Path("requirement.txt")
if REQ_FILE.exists():
    try:
        import torch
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])
if os.name == "nt":
    try:
        import torch_directml
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-directml"])

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import food_cv.training as training_module
importlib.reload(training_module)
from food_cv.classifier import build_resnet50_classifier, unfreeze_last_two_blocks
from food_cv.config import ProjectPaths
from food_cv.data_pipeline import DataConfig, Food101DataModule
from food_cv.evaluation import evaluate_classification_metrics, evaluate_nutrition_hit_rate
from food_cv.pipeline import MealPredictor
from food_cv.training import TrainConfig, train_classifier_two_stage

paths = ProjectPaths.from_root(PROJECT_ROOT)
candidate_data_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "food101_data",
    Path("/content/data"),
    Path("/content/datasets"),
    PROJECT_ROOT,
]
data_root = next((p for p in candidate_data_roots if p.exists()), PROJECT_ROOT)
data_config = DataConfig(
    data_root=data_root,
    batch_size=16,
    num_workers=0,
    image_size=224,
    download_if_missing=True,
)

print("[main] 初始化数据模块", flush=True)
try:
    datamodule = Food101DataModule(data_config)
    print("[main] 构建 DataLoader", flush=True)
    train_loader, val_loader, test_loader = datamodule.build_dataloaders()
    class_names = datamodule.get_class_names()
    print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
    print(f"labels={len(class_names)}")
except Exception as e:
    train_loader = val_loader = test_loader = None
    class_names = []
    print(f"Data loader 初始化失败: {e}")

model = build_resnet50_classifier(num_classes=101, freeze_backbone=True)
stage1_checkpoint = paths.model_dir / "food101_resnet50_stage1.pt"
stage2_checkpoint = paths.model_dir / "food101_resnet50_stage2.pt"

if train_loader is not None and val_loader is not None:
    print("[main] 开始训练", flush=True)
    metrics = train_classifier_two_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        stage1_save_path=stage1_checkpoint,
        stage1_config=TrainConfig(
            epochs=1,
            lr=1e-4,
            device="directml" if os.name == "nt" else "auto",
            log_every_n_steps=5,
            max_steps_per_epoch=60,
            use_amp=True,
            require_accelerator=True,
            optimizer_foreach=False,
        ),
        stage2_save_path=stage2_checkpoint,
        stage2_config=TrainConfig(
            epochs=1,
            lr=3e-5,
            device="directml" if os.name == "nt" else "auto",
            log_every_n_steps=5,
            max_steps_per_epoch=60,
            use_amp=True,
            require_accelerator=True,
            optimizer_foreach=False,
        ),
        unfreeze_fn=unfreeze_last_two_blocks,
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    quick_eval = evaluate_classification_metrics(
        model=model,
        loader=test_loader,
        device=model.fc.weight.device,
        max_batches=80,
    )
    print("[main] quick test metrics (sampled):")
    print(json.dumps(quick_eval, ensure_ascii=False, indent=2))
else:
    print("跳过训练：请确认 Food-101 已下载并放到 data 目录")

demo_image = PROJECT_ROOT / "demo.jpg"
if stage2_checkpoint.exists() and demo_image.exists():
    predictor = MealPredictor(paths=paths, checkpoint_path=stage2_checkpoint, labels=class_names if class_names else None)
    result = predictor.predict_meal(demo_image)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    nutrition_eval = evaluate_nutrition_hit_rate(predictor, [demo_image])
    print("[main] nutrition hit metrics:")
    print(json.dumps(nutrition_eval, ensure_ascii=False, indent=2))
    if result.get("total", {}).get("calories", 0.0) <= 0.0:
        print("[main] 警告：营养命中率偏低，请继续扩展 Food-101->USDA 映射", flush=True)
elif not stage2_checkpoint.exists():
    print("未找到 Stage2 checkpoint，已跳过推理。请先完成两阶段训练。")
else:
    print("未找到 demo.jpg，推理演示已跳过")


[main] 初始化数据模块
[main] 构建 DataLoader
train=75750 val=25250 test=25250
labels=101
[main] 开始训练
[train] backend=directml device=privateuseone:0
[train] epoch=1/1 start
[train] epoch_loop start total_steps=60
[train] first_batch_loaded
[train] step=5/60 avg_loss=4.6194 speed=9.85step/s eta=0.1min
[train] step=10/60 avg_loss=4.6245 speed=9.84step/s eta=0.1min
[train] step=15/60 avg_loss=4.6223 speed=9.94step/s eta=0.1min
[train] step=20/60 avg_loss=4.6181 speed=10.03step/s eta=0.1min
[train] step=25/60 avg_loss=4.6170 speed=10.06step/s eta=0.1min
[train] step=30/60 avg_loss=4.6200 speed=10.11step/s eta=0.0min
[train] step=35/60 avg_loss=4.6205 speed=10.07step/s eta=0.0min
[train] step=40/60 avg_loss=4.6248 speed=9.99step/s eta=0.0min
[train] step=45/60 avg_loss=4.6213 speed=9.94step/s eta=0.0min
[train] step=50/60 avg_loss=4.6136 speed=9.90step/s eta=0.0min
[train] step=55/60 avg_loss=4.6122 speed=9.88step/s eta=0.0min
[train] step=60/60 avg_loss=4.6109 speed=9.90step/s eta=0.0min
[train] ep

100%|██████████| 6.25M/6.25M [00:00<00:00, 7.44MB/s]


{
  "image_path": "d:\\CityU\\Projects\\CV_course_project\\demo.jpg",
  "top3_classification": [
    {
      "label": "greek_salad",
      "confidence": 0.01612081006169319
    },
    {
      "label": "tuna_tartare",
      "confidence": 0.015825802460312843
    },
    {
      "label": "bruschetta",
      "confidence": 0.015387032181024551
    }
  ],
  "items": [
    {
      "label": "greek_salad",
      "confidence": 0.8558221459388733,
      "weight_g": 143.96130909958262,
      "calories": 0.0,
      "protein_g": 0.0,
      "fat_g": 0.0,
      "carbs_g": 0.0
    },
    {
      "label": "tuna_tartare",
      "confidence": 0.5971248149871826,
      "weight_g": 57.63236798433121,
      "calories": 0.0,
      "protein_g": 0.0,
      "fat_g": 0.0,
      "carbs_g": 0.0
    },
    {
      "label": "bruschetta",
      "confidence": 0.5770450234413147,
      "weight_g": 92.97515164518357,
      "calories": 0.0,
      "protein_g": 0.0,
      "fat_g": 0.0,
      "carbs_g": 0.0
    }
  ],
  "tot

In [6]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
REPO_OWNER = 'Blank4cc'
REPO_NAME = 'VisualDietician'
if not GITHUB_TOKEN:
    raise RuntimeError('请先在 Colab 环境变量中设置 GITHUB_TOKEN')
PRIVATE_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
DEST_PATH = f'/content/drive/MyDrive/{REPO_NAME}'

if not os.path.exists(DEST_PATH):
    print('正在尝试克隆私有仓库...')
    !git clone {PRIVATE_REPO_URL} {DEST_PATH}
else:
    print(f'目录已存在: {DEST_PATH}')

if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    print(f'成功进入目录: {os.getcwd()}')
    !ls
else:
    print('克隆仍然失败，请检查 Token 权限或仓库路径是否正确。')
Path(DEST_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在尝试克隆私有仓库...
Cloning into '/content/drive/MyDrive/VisualDietician'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 75 (delta 12), reused 61 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 18.03 MiB | 10.44 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Updating files: 100% (57/57), done.
成功进入目录: /content/drive/MyDrive/VisualDietician
food_data  main.ipynb  src  workflow


In [7]:
import os

# 确保你在项目目录下
if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    # 使用 git pull 拉取最新代码
    !git pull
    print("代码已尝试更新。")
else:
    print(f"未找到目录 {DEST_PATH}，请先运行克隆单元格。")

Already up to date.
代码已尝试更新。


In [8]:
import sys
from pathlib import Path

# 将当前路径（包含 src 目录的路径）加入 sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print("当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。")

当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。
